In [0]:
select
  *
from
  read_files('/Volumes/idp/default/final_project')

In [0]:
-- parse the content
create or replace table parsed_data as 
select
  path,
  ai_parse_document(content) as parsed_content
from
  read_files('/Volumes/idp/default/final_project')

In [0]:
create or replace table pretty_data as 
select
  path,
  concat_ws('/n',
    transform(try_cast(parsed_content:document:elements as array<variant>),
    e -> coalesce(try_cast(e:content as string), ''))
  ) as doc_text
from 
  parsed_data

In [0]:
-- classify the documents
create or replace table classified_data as 
select
  *,
  ai_classify(doc_text, array('Invoice', 'Purchase Order', 'Reciept', 'Other')) as doc_classification
from
  pretty_data

extract and store data from invoice documents

In [0]:
-- extract the data from the invoices table and put them into a separate table
create or replace table invoice_data as
select
  *,
  ai_extract(doc_text,
    array('Vendor_Name', 'Invoice_Number', 'Invoice_Date', 'Due_Date', 'Payment_Method', 'Total')) as extracted
from
  classified_data
where doc_classification = 'Invoice'

In [0]:
-- create the schema first 
create schema if not exists idp.finance

In [0]:
-- create a table with only the document data
create or replace table idp.finance.invoices as
select
  path,
  extracted.Vendor_Name as Vendor,
  extracted.Invoice_Number as Invoice_Number,
  extracted.Invoice_Date as Invoice_Date,
  extracted.Due_Date as Due_Date,
  extracted.Payment_Method as Payment_Method,
  extracted.Total as Total
from
  invoice_data

In [0]:
select * from idp.finance.invoices

repeat for purchase orders

In [0]:
-- extract the data from the purchases table and put them into a separate table
create or replace table purchase_order_data as
select
  *,
  ai_extract(doc_text,
    array('Merchant_Name', 'PO_Number', 'Purchase_Order_Date', 'Total')) as extracted
from
  classified_data
where doc_classification = 'Purchase Order'

In [0]:
create or replace table idp.finance.purchase_order as
select
  path,
  extracted.Merchant_Name as Merchant_Name,
  extracted.PO_Number as PO_Number,
  extracted.Purchase_Order_Date as Purchase_Order_Date,
  extracted.Total as Total
from
  purchase_order_data

In [0]:
select * from idp.finance.purchase_order

repeat for receipts

In [0]:
-- extract the data from the purchases table and put them into a separate table
create or replace table reciept_data as
select
  *,
  ai_extract(doc_text,
    array('Merchant_Name', 'Reciept_Number', 'Transaction_Date', 'Total')) as extracted
from
  classified_data
where doc_classification = 'Reciept'

In [0]:
create or replace table idp.finance.reciepts as
select
  path,
  extracted.Merchant_Name as Merchant_Name,
  extracted.Reciept_Number as Reciept_Number,
  extracted.Transaction_Date as Transaction_Date,
  extracted.Total as Total
from
  reciept_data

In [0]:
select * from idp.finance.reciepts